# LeetCode 693 — Binary Number with Alternating Bits bằng Python

**Link đề:** https://leetcode.com/problems/binary-number-with-alternating-bits/
**Độ khó:** Easy · **Chủ đề:** Bit Manipulation

```python
class Solution:
    def hasAlternatingBits(self, n: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu **biểu diễn nhị phân** của số nguyên và hàm `bin()`.
2. Tách từng bit bằng `n % 2` / `n // 2` và bằng toán tử bit `n & 1` / `n >> 1`.
3. Giải bài bằng chuỗi (`bin(n)`, `"00" in s`) rồi bằng vòng lặp bit `O(1)` bộ nhớ.
4. Hiểu mẹo **XOR với chính nó dịch phải**: `x = n ^ (n >> 1)` toàn bit 1 ⇔ bit xen kẽ.
5. Biết kiểm tra "toàn bit 1" bằng `x & (x + 1) == 0`.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho số nguyên **dương** `n`. Kiểm tra biểu diễn nhị phân của `n` có **bit xen kẽ** không: hai bit **kề nhau luôn khác nhau**.

| `n` | nhị phân | Kết quả |
|---:|---|---|
| 5 | `101` | `True` |
| 7 | `111` | `False` |
| 11 | `1011` | `False` (có `11` ở cuối) |
| 10 | `1010` | `True` |
| 1 | `1` | `True` (chỉ một bit) |

**Constraints:** `1 <= n <= 2^31 - 1`.

Chỉ xét các bit **có nghĩa** (từ bit 1 cao nhất trở xuống) — không tính các số 0 vô nghĩa ở đầu.

## 2. Nền tảng Python cần biết

### 2.1. Hệ nhị phân và `bin()`
Mỗi bit là một lũy thừa của 2: `1011₂ = 1·8 + 0·4 + 1·2 + 1·1 = 11`.
`bin(n)` trả chuỗi có tiền tố `"0b"`; dùng `bin(n)[2:]` hoặc `format(n, "b")` để bỏ tiền tố.

In [1]:
for n in [5, 7, 11, 10, 1]:
    print(f"{n:>3} -> bin = {bin(n):<8} -> bỏ '0b' = {bin(n)[2:]:<6} | format = {format(n, 'b')}")
print("kiểm tra 1011 -> ", 1*8 + 0*4 + 1*2 + 1*1, "| int('1011', 2) =", int("1011", 2))

  5 -> bin = 0b101    -> bỏ '0b' = 101    | format = 101
  7 -> bin = 0b111    -> bỏ '0b' = 111    | format = 111
 11 -> bin = 0b1011   -> bỏ '0b' = 1011   | format = 1011
 10 -> bin = 0b1010   -> bỏ '0b' = 1010   | format = 1010
  1 -> bin = 0b1      -> bỏ '0b' = 1      | format = 1
kiểm tra 1011 ->  11 | int('1011', 2) = 11


### 2.2. Lấy bit cuối và bỏ bit cuối
Giống `% 10`, `// 10` cho chữ số thập phân:
- `n % 2` hoặc `n & 1` → **bit cuối** (0 hoặc 1).
- `n // 2` hoặc `n >> 1` → **bỏ bit cuối** (dịch phải 1 bit).

In [2]:
n = 11
while n > 0:
    print(f"n={n:>2} ({format(n, 'b'):>4}) -> bit cuối: n % 2 = {n % 2}, n & 1 = {n & 1} | n >> 1 = {n >> 1}")
    n >>= 1

n=11 (1011) -> bit cuối: n % 2 = 1, n & 1 = 1 | n >> 1 = 5
n= 5 ( 101) -> bit cuối: n % 2 = 1, n & 1 = 1 | n >> 1 = 2
n= 2 (  10) -> bit cuối: n % 2 = 0, n & 1 = 0 | n >> 1 = 1
n= 1 (   1) -> bit cuối: n % 2 = 1, n & 1 = 1 | n >> 1 = 0


### 2.3. Các toán tử bit
| Toán tử | Tên | Ví dụ (`a=1010₂=10`, `b=0110₂=6`) |
|---|---|---|
| `a & b` | AND — 1 khi **cả hai** là 1 | `0010₂ = 2` |
| `a \| b` | OR — 1 khi **ít nhất một** là 1 | `1110₂ = 14` |
| `a ^ b` | XOR — 1 khi **khác nhau** | `1100₂ = 12` |
| `a >> k` | dịch phải `k` bit (≈ chia `2^k`) | `10 >> 1 = 5` |
| `a << k` | dịch trái `k` bit (≈ nhân `2^k`) | `10 << 1 = 20` |

In [3]:
a, b2 = 0b1010, 0b0110
for name, v in [("a & b", a & b2), ("a | b", a | b2), ("a ^ b", a ^ b2), ("a >> 1", a >> 1), ("a << 1", a << 1)]:
    print(f"{name:<7} = {v:>3} = {format(v, '05b')}")

a & b   =   2 = 00010
a | b   =  14 = 01110
a ^ b   =  12 = 01100
a >> 1  =   5 = 00101
a << 1  =  20 = 10100


### 2.4. Kiểm tra chuỗi con bằng `in`
`"00" in s` trả `True` nếu chuỗi `s` chứa `"00"` ở đâu đó.

In [4]:
s = bin(11)[2:]
print(s, "| có '11'?", "11" in s, "| có '00'?", "00" in s)

1011 | có '11'? True | có '00'? False


## 3. Cách trực giác — đổi sang chuỗi, so từng cặp kề nhau

1. `s = bin(n)[2:]`.
2. Với mọi `i` từ 1 đến `len(s)-1`, nếu `s[i] == s[i-1]` → `False`.
3. Không có cặp nào trùng → `True`.

Số bit của `n ≤ 2^31 - 1` tối đa 31 → rất nhanh. Cách này tốn thêm `O(log n)` bộ nhớ cho chuỗi.

In [5]:
def alt_bits_string(n: int) -> bool:
    s = bin(n)[2:]
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            return False
    return True

for n in [5, 7, 11, 10, 1, 2, 3]:
    print(n, format(n, "b"), "->", alt_bits_string(n))

5 101 -> True
7 111 -> False
11 1011 -> False
10 1010 -> True
1 1 -> True
2 10 -> True
3 11 -> False


### 3.1. Phiên bản 1 dòng: không chứa `"00"` và không chứa `"11"`
Hai bit kề nhau trùng nhau ⇔ chuỗi chứa `"00"` hoặc `"11"`.

In [6]:
def alt_bits_substring(n: int) -> bool:
    s = bin(n)
    return "00" not in s and "11" not in s

print([alt_bits_substring(n) for n in [5, 7, 11, 10, 1]])

[True, False, False, True, True]


Lưu ý: ở trên dùng `bin(n)` **có** tiền tố `"0b"`. Có an toàn không? Tiền tố là `"0b"` rồi đến bit đầu luôn là `1` → chuỗi bắt đầu `"0b1..."`: không tạo ra `"00"` hay `"11"` giả. Vẫn đúng, nhưng viết `bin(n)[2:]` rõ ràng hơn.

## 4. Nút thắt và insight

Cách chuỗi đã đủ nhanh. "Nút thắt" ở đây là **bộ nhớ và tính thuần bit**: phỏng vấn viên thường muốn lời giải **không tạo chuỗi**.

**Insight 1 — vòng lặp bit:** giữ `prev = n & 1`, mỗi bước dịch phải `n >>= 1`, so bit cuối mới `cur = n & 1` với `prev`.

**Insight 2 — XOR một phát:** nếu bit xen kẽ, dịch phải 1 bit tạo ra số có mọi bit **ngược** với `n` ở cùng vị trí:
```text
n        = 1 0 1 0 1   (21)
n >> 1   = 0 1 0 1 0   (10)
n ^ (n>>1)= 1 1 1 1 1   (31)  -> toàn bit 1!
```
Số "toàn bit 1" có dạng `2^k - 1`; cộng thêm 1 thành `2^k` (một bit 1 rồi toàn 0) → `x & (x + 1) == 0`.

In [7]:
for n in [21, 10, 11, 7]:
    x = n ^ (n >> 1)
    print(f"n={n:>2} {format(n, '06b')} | n>>1 {format(n >> 1, '06b')} | x {format(x, '06b')} "
          f"| x+1 {format(x + 1, '06b')} | x&(x+1) = {x & (x + 1)}")

n=21 010101 | n>>1 001010 | x 011111 | x+1 100000 | x&(x+1) = 0
n=10 001010 | n>>1 000101 | x 001111 | x+1 010000 | x&(x+1) = 0
n=11 001011 | n>>1 000101 | x 001110 | x+1 001111 | x&(x+1) = 14
n= 7 000111 | n>>1 000011 | x 000100 | x+1 000101 | x&(x+1) = 4


## 5. Lời giải chính — vòng lặp bit (giảng từng đoạn)

**Đoạn 1:** `prev = n & 1` (bit cuối), `n >>= 1`.

**Đoạn 2:** trong khi `n > 0`:
- `cur = n & 1`
- nếu `cur == prev` → hai bit kề nhau trùng → `False`.
- `prev = cur`, `n >>= 1`.

**Đoạn 3:** hết bit có nghĩa mà không trùng → `True`.

Vòng `while n > 0` dừng đúng lúc hết các bit có nghĩa → không xét số 0 vô nghĩa ở đầu.

In [8]:
def alt_bits_loop(n: int) -> bool:
    prev = n & 1                 # đoạn 1
    n >>= 1
    while n > 0:                 # đoạn 2
        cur = n & 1
        if cur == prev:
            return False
        prev = cur
        n >>= 1
    return True                  # đoạn 3

for n in [5, 7, 11, 10, 1]:
    print(n, "->", alt_bits_loop(n))

5 -> True
7 -> False
11 -> False
10 -> True
1 -> True


## 6. TRACE từng bước

`n = 21` (`10101₂`) và `n = 11` (`1011₂`):

| Bước | `n` (nhị phân) | `cur = n & 1` | `prev` | Quyết định |
|---:|---|---:|---:|---|
| khởi tạo | 10101 | – | 1 | `n >>= 1` → 1010 |
| 1 | 1010 | 0 | 1 | khác → tiếp |
| 2 | 101 | 1 | 0 | khác → tiếp |
| 3 | 10 | 0 | 1 | khác → tiếp |
| 4 | 1 | 1 | 0 | khác → tiếp, `n` → 0 → `True` |

Với `11 = 1011₂`: khởi tạo `prev = 1`, `n = 101`; bước 1: `cur = 1 == prev` → `False` ngay.

In [9]:
def trace_alt_bits(n: int) -> bool:
    print(f"n = {n} = {format(n, 'b')}")
    prev = n & 1
    n >>= 1
    print(f"  khởi tạo: prev = {prev}, n -> {format(n, 'b') if n else '0'}")
    step = 1
    while n > 0:
        cur = n & 1
        same = cur == prev
        print(f"  bước {step}: n = {format(n, 'b'):>5}, cur = {cur}, prev = {prev} -> {'TRÙNG -> False' if same else 'khác'}")
        if same:
            return False
        prev = cur
        n >>= 1
        step += 1
    print("  hết bit -> True")
    return True

trace_alt_bits(21)
print()
trace_alt_bits(11)

n = 21 = 10101
  khởi tạo: prev = 1, n -> 1010
  bước 1: n =  1010, cur = 0, prev = 1 -> khác
  bước 2: n =   101, cur = 1, prev = 0 -> khác
  bước 3: n =    10, cur = 0, prev = 1 -> khác
  bước 4: n =     1, cur = 1, prev = 0 -> khác
  hết bit -> True

n = 11 = 1011
  khởi tạo: prev = 1, n -> 101
  bước 1: n =   101, cur = 1, prev = 1 -> TRÙNG -> False


False

In [10]:
trace_alt_bits(1)
print()
trace_alt_bits(4)

n = 1 = 1
  khởi tạo: prev = 1, n -> 0
  hết bit -> True

n = 4 = 100
  khởi tạo: prev = 0, n -> 10
  bước 1: n =    10, cur = 0, prev = 0 -> TRÙNG -> False


False

## 7. Code nộp LeetCode

### 7.1. Cách 1 — Vòng lặp bit (khuyên dùng để luyện bit)

In [11]:
class Solution:
    def hasAlternatingBits(self, n: int) -> bool:
        prev = n & 1
        n >>= 1
        while n:
            cur = n & 1
            if cur == prev:
                return False
            prev = cur
            n >>= 1
        return True

print([Solution().hasAlternatingBits(n) for n in [5, 7, 11]])

[True, False, False]


### 7.2. Cách 2 — Chuỗi (ngắn nhất)

In [12]:
class SolutionString:
    def hasAlternatingBits(self, n: int) -> bool:
        s = bin(n)[2:]
        return "00" not in s and "11" not in s

print([SolutionString().hasAlternatingBits(n) for n in [5, 7, 11]])

[True, False, False]


### 7.3. Cách 3 — XOR + kiểm tra toàn bit 1 (`O(1)`)

In [13]:
class SolutionXor:
    def hasAlternatingBits(self, n: int) -> bool:
        x = n ^ (n >> 1)
        return (x & (x + 1)) == 0

print([SolutionXor().hasAlternatingBits(n) for n in [5, 7, 11]])

[True, False, False]


### 7.4. Cách 4 — So với "mẫu" đã biết
Các số bit xen kẽ có dạng `1`, `10`, `101`, `1010`, ... Với tối đa 31 bit chỉ có **31** số như vậy (mỗi độ dài `1..31` đúng một số, vì bit đầu luôn là 1 và các bit sau bị ép xen kẽ) → dựng sẵn `set` rồi tra. (Mẹo "tiền tính toán" — hữu ích khi đáp án hợp lệ rất ít.)

In [14]:
# Mỗi bước nối thêm vào cuối một bit NGƯỢC với bit cuối hiện tại
def build_alternating_set_clear(max_bits: int = 31) -> set:
    res = set()
    v = 1                                    # "1"
    while v.bit_length() <= max_bits:
        res.add(v)
        v = (v << 1) | (1 - (v & 1))         # nối bit ngược với bit cuối: 1 -> 10 -> 101 -> 1010 ...
    return res

ALT = build_alternating_set_clear()
print("số lượng:", len(ALT), "| vài số nhỏ:", sorted(ALT)[:8])

class SolutionLookup:
    def hasAlternatingBits(self, n: int) -> bool:
        return n in ALT

print([SolutionLookup().hasAlternatingBits(n) for n in [5, 7, 11]])

số lượng: 31 | vài số nhỏ: [1, 2, 5, 10, 21, 42, 85, 170]
[True, False, False]


## 8. Vì sao đúng?

**Vòng lặp bit — bất biến:** ở đầu mỗi vòng, `prev` là bit vừa bị dịch ra (bit ở vị trí thấp hơn), `n & 1` là bit kế tiếp phía trên. So hai bit kề nhau theo thứ tự từ thấp lên cao; mọi cặp kề nhau trong các bit có nghĩa đều được so đúng một lần. Vòng dừng khi `n == 0`, tức đã đi qua bit 1 cao nhất.

**XOR:** bit thứ `i` của `x = n ^ (n >> 1)` là `b_i XOR b_{i+1}` (với `b_{i+1} = 0` khi vượt bit cao nhất). Bit xen kẽ ⇔ mọi `b_i ≠ b_{i+1}` cho `i` từ 0 đến bit cao nhất − 1 ⇔ các bit đó của `x` đều là 1; còn bit cao nhất của `x` là `1 XOR 0 = 1`. Vậy `x = 2^k - 1` (toàn bit 1) ⇔ bit xen kẽ.

`x` toàn bit 1 ⇔ `x + 1` là lũy thừa của 2 và không chung bit nào với `x` ⇔ `x & (x + 1) == 0`.

## 9. Độ phức tạp và so sánh

Gọi `b = số bit của n` (≤ 31), tức `b = O(log n)`.

| Cách | Time | Space | Ghi chú |
|---|---:|---:|---|
| Chuỗi, so cặp | `O(b)` | `O(b)` | Dễ hiểu nhất |
| Chuỗi, `"00"`/`"11"` | `O(b)` | `O(b)` | Một dòng |
| Vòng lặp bit | `O(b)` | `O(1)` | Luyện bit |
| XOR + `x & (x+1)` | `O(1)` | `O(1)` | Mẹo đẹp |
| Tra bảng 31 số | `O(1)` | `O(1)` (31 phần tử) | Tiền tính toán |

Với `n ≤ 2^31 - 1`, mọi cách đều coi như hằng số; khác biệt chủ yếu là **kỹ thuật**.

## 10. Bộ test

1. Ba ví dụ đề: 5, 7, 11.
2. Biên: `1`, `2` (`10`), `3` (`11`), `2^31 - 1` (toàn bit 1).
3. Số xen kẽ lớn nhất ≤ `2^31 - 1`: `1010...1` (31 bit) và `0b1010...10` (30 bit).
4. Lũy thừa của 2 lớn hơn 2 (`100`, `1000`) → `False`.
5. Vét cạn `1..5000`: so mọi cách.

In [15]:
big_alt_31 = int("10" * 15 + "1", 2)     # 31 bit: 1010...101
big_alt_30 = int("10" * 15, 2)           # 30 bit: 1010...10
cases = [
    (5, True), (7, False), (11, False), (10, True),
    (1, True), (2, True), (3, False), (4, False), (8, False),
    (2**31 - 1, False), (big_alt_31, True), (big_alt_30, True), (big_alt_31 - 2, False),
]
solvers = [Solution().hasAlternatingBits, SolutionString().hasAlternatingBits, SolutionXor().hasAlternatingBits,
           SolutionLookup().hasAlternatingBits, alt_bits_string, alt_bits_loop, alt_bits_substring]
for n, expected in cases:
    for f in solvers:
        assert f(n) == expected, (f, n)
    print(f"{n:>11} = {format(n, 'b'):<32} -> {expected} PASS")

for n in range(1, 5001):
    exp = alt_bits_string(n)
    assert all(f(n) == exp for f in solvers), n
print("Vét cạn 1..5000: mọi cách khớp PASS")
print("Các số bit xen kẽ ≤ 100:", [n for n in range(1, 101) if alt_bits_loop(n)])

          5 = 101                              -> True PASS
          7 = 111                              -> False PASS
         11 = 1011                             -> False PASS
         10 = 1010                             -> True PASS
          1 = 1                                -> True PASS
          2 = 10                               -> True PASS
          3 = 11                               -> False PASS
          4 = 100                              -> False PASS
          8 = 1000                             -> False PASS
 2147483647 = 1111111111111111111111111111111  -> False PASS
 1431655765 = 1010101010101010101010101010101  -> True PASS
  715827882 = 101010101010101010101010101010   -> True PASS
 1431655763 = 1010101010101010101010101010011  -> False PASS
Vét cạn 1..5000: mọi cách khớp PASS
Các số bit xen kẽ ≤ 100: [1, 2, 5, 10, 21, 42, 85]


## 11. Lỗi người mới hay gặp

### Lỗi 1: Quên bỏ tiền tố `"0b"` khi so cặp ký tự
`bin(5) = "0b101"`: cặp `('0', 'b')` không sao, nhưng nếu so **theo chỉ số cố định** (ví dụ `s[0]` là bit cao nhất) thì sai. Với kiểu so cặp kề nhau, `'b'` khác mọi bit nên "may mắn" không sai — đừng dựa vào may mắn.

### Lỗi 2: Dùng `n / 2` thay vì `n // 2` / `n >> 1`
`/` cho ra `float` → `5 / 2 = 2.5`, `2.5 % 2 = 0.5` → logic bit hỏng, vòng lặp có thể không dừng như mong đợi.

In [16]:
print("bin(5) =", bin(5), "| s[0] =", bin(5)[0], "(không phải bit cao nhất!)")
print("5 / 2 =", 5 / 2, "| 5 // 2 =", 5 // 2, "| 5 >> 1 =", 5 >> 1)
print("(5 / 2) % 2 =", (5 / 2) % 2, "-> không phải 0/1")

bin(5) = 0b101 | s[0] = 0 (không phải bit cao nhất!)
5 / 2 = 2.5 | 5 // 2 = 2 | 5 >> 1 = 2
(5 / 2) % 2 = 0.5 -> không phải 0/1


### Lỗi 3: Kiểm tra "toàn bit 1" bằng `x & (x - 1) == 0`
`x & (x - 1) == 0` là kiểm tra **lũy thừa của 2** (đúng một bit 1), không phải toàn bit 1. Phải dùng `x & (x + 1)`.

In [17]:
def wrong_power_check(n):
    x = n ^ (n >> 1)
    return x & (x - 1) == 0          # SAI: kiểm tra lũy thừa của 2

for n in [5, 10, 4, 6]:
    print(f"n={n} ({format(n, 'b')}): sai = {wrong_power_check(n)}, đúng = {Solution().hasAlternatingBits(n)}")

n=5 (101): sai = False, đúng = True
n=10 (1010): sai = False, đúng = True
n=4 (100): sai = False, đúng = False
n=6 (110): sai = False, đúng = False


### Lỗi 4: Mang thói quen ưu tiên toán tử từ C/Java sang (và ngược lại)
Thứ tự ưu tiên trong **Python**: `+ -` > `<< >>` > `&` > `^` > `|` > so sánh (`==`, `<`...).
Vì vậy trong Python `x & x + 1 == 0` được hiểu là `(x & (x + 1)) == 0` — **đúng ý**.
Nhưng trong **C/C++/Java**, `==` ưu tiên **cao hơn** `&`, nên `x & (x + 1) == 0` lại thành `x & ((x + 1) == 0)` — sai!
Thói quen tốt cho mọi ngôn ngữ: **luôn thêm ngoặc** `(x & (x + 1)) == 0`. Cell dưới kiểm chứng trong Python ba cách viết cho cùng kết quả.

In [18]:
x = 0b11111
print("x & x + 1 == 0     ->", x & x + 1 == 0)
print("x & (x + 1) == 0   ->", x & (x + 1) == 0)
print("(x & (x + 1)) == 0 ->", (x & (x + 1)) == 0)
x = 0b10111
print("x = 10111 (không toàn bit 1):", x & x + 1 == 0, x & (x + 1) == 0, (x & (x + 1)) == 0)

x & x + 1 == 0     -> True
x & (x + 1) == 0   -> True
(x & (x + 1)) == 0 -> True
x = 10111 (không toàn bit 1): False False False


## 12. Mẹo nhận dạng pattern

- **"Biểu diễn nhị phân / bit"** → nghĩ ngay tới `n & 1`, `n >> 1`, `bin(n)`, `n.bit_count()` (Python 3.10+).
- **"So bit kề nhau"** → `n ^ (n >> 1)` (mỗi bit của kết quả cho biết hai bit kề nhau có khác nhau không). Đây cũng chính là công thức **mã Gray**.
- Các mẫu kiểm tra nhanh:
  - lũy thừa của 2: `n > 0 and n & (n - 1) == 0`
  - toàn bit 1: `(n & (n + 1)) == 0`
  - bit thấp nhất bằng 1: `n & -n`
- Bài cùng họ: 191 Number of 1 Bits, 231 Power of Two, 338 Counting Bits, 476 Number Complement, 89 Gray Code.

## 13. Biến thể

### 13.1. Đếm số nguyên trong `[1, N]` có bit xen kẽ
Dùng bảng `ALT` đã dựng.

In [19]:
def count_alternating_upto(N: int) -> int:
    return sum(1 for v in ALT if v <= N)

print(count_alternating_upto(10), count_alternating_upto(100), count_alternating_upto(2**31 - 1))

4 7 31


### 13.2. Mã Gray (LeetCode 89)
Mã Gray thứ `i` là `i ^ (i >> 1)` — hai mã liên tiếp chỉ khác **đúng 1 bit**.

In [20]:
def gray_code(k: int) -> list[int]:
    return [i ^ (i >> 1) for i in range(1 << k)]

g = gray_code(3)
print(g, [format(v, "03b") for v in g])
print("mỗi cặp liên tiếp khác đúng 1 bit:", all(bin(a ^ b2).count("1") == 1 for a, b2 in zip(g, g[1:])))

[0, 1, 3, 2, 6, 7, 5, 4] ['000', '001', '011', '010', '110', '111', '101', '100']
mỗi cặp liên tiếp khác đúng 1 bit: True


## 14. Bài tập (tự làm trước, đáp án ở cuối)

**Bài 1 — Tính tay:** `42` ở hệ nhị phân là gì? Có bit xen kẽ không? Tính `42 ^ (42 >> 1)`.

**Bài 2 — Đoán output:** `print(13 & 1, 13 >> 2, 6 ^ 3, bin(10)[2:])`.

**Bài 3 — TODO:** viết `exercise_alt_bits_mod` dùng **chỉ** `% 2` và `// 2` (không dùng toán tử bit, không dùng chuỗi).

In [21]:
def exercise_alt_bits_mod(n: int) -> bool:
    # TODO: prev = n % 2; n //= 2; while n > 0: ...
    pass

# assert exercise_alt_bits_mod(5) is True
# assert exercise_alt_bits_mod(11) is False

**Bài 4 — TODO:** trả về **độ dài đoạn bit xen kẽ dài nhất** (liên tiếp) trong biểu diễn nhị phân của `n`.

In [22]:
def exercise_longest_alternating_run(n: int) -> int:
    # TODO: duyệt bin(n)[2:], đếm chuỗi chạy: cur + 1 nếu khác bit trước, ngược lại 1
    pass

# assert exercise_longest_alternating_run(0b1011010) == 4     # "1011010" -> đoạn "1010" ở cuối
# assert exercise_longest_alternating_run(0b10101) == 5

**Bài 5 — TODO (LeetCode 476):** trả về **số bù** của `n` — lật mọi bit có nghĩa (`5 = 101` → `010 = 2`).

In [23]:
def exercise_complement(n: int) -> int:
    # TODO: mask = (1 << n.bit_length()) - 1; return n ^ mask
    pass

# assert exercise_complement(5) == 2
# assert exercise_complement(1) == 0

**Bài 6 — Mini challenge:** cho `n`, trả về **số bit xen kẽ nhỏ nhất `≥ n`**.

In [24]:
def exercise_next_alternating(n: int) -> int:
    # TODO: dùng bảng ALT (sắp xếp) hoặc tự sinh các số xen kẽ theo số bit
    pass

# assert exercise_next_alternating(5) == 5
# assert exercise_next_alternating(6) == 10
# assert exercise_next_alternating(11) == 21

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `42 = 32 + 8 + 2` | `101010` | XOR với `10101` |
| 2 | `13 = 1101` | `13 >> 2 = 11₂` | `6 ^ 3 = 110 ^ 011` |
| 3 | `% 2` = bit cuối | `// 2` = bỏ bit cuối | giống `alt_bits_loop` |
| 4 | Giống bài 674 (đếm chuỗi chạy) | Điều kiện nối: `s[i] != s[i-1]` | `best = max(best, cur)` |
| 5 | Mặt nạ toàn bit 1 dài bằng `n` | `n.bit_length()` | `n ^ mask` |
| 6 | Có rất ít số xen kẽ | `sorted(ALT)` | Số đầu tiên `>= n` |

## 16. Đáp án tham khảo

**Bài 1:** `42 = 101010₂` → xen kẽ ✔. `42 >> 1 = 10101₂ = 21`, `42 ^ 21 = 111111₂ = 63` (toàn bit 1).

**Bài 2:** `13 & 1 = 1`, `13 >> 2 = 3`, `6 ^ 3 = 5`, `bin(10)[2:] = "1010"` → in `1 3 5 1010`.

**Bài 4:** `0b1011010` = `1011010`: đi từ trái, `1-0-1` (dài 3), gặp `1-1` thì gãy (reset 1), rồi `1-0-1-0` (dài 4) → đáp án **4**. Với `10101` cả chuỗi xen kẽ → **5**.

In [25]:
print("Bài 1:", format(42, "b"), Solution().hasAlternatingBits(42), 42 ^ (42 >> 1), format(42 ^ (42 >> 1), "b"))
print("Bài 2:", 13 & 1, 13 >> 2, 6 ^ 3, bin(10)[2:])

Bài 1: 101010 True 63 111111
Bài 2: 1 3 5 1010


In [26]:
def answer_alt_bits_mod(n):
    prev = n % 2
    n //= 2
    while n > 0:
        cur = n % 2
        if cur == prev:
            return False
        prev = cur
        n //= 2
    return True


def answer_longest_alternating_run(n):
    s = bin(n)[2:]
    best = cur = 1
    for i in range(1, len(s)):
        cur = cur + 1 if s[i] != s[i - 1] else 1
        best = max(best, cur)
    return best


def answer_complement(n):
    mask = (1 << n.bit_length()) - 1
    return n ^ mask


def answer_next_alternating(n):
    for v in sorted(ALT):
        if v >= n:
            return v
    return -1


assert answer_alt_bits_mod(5) is True
assert answer_alt_bits_mod(11) is False
assert answer_longest_alternating_run(0b1011010) == 4
assert answer_longest_alternating_run(0b10101) == 5
assert answer_complement(5) == 2
assert answer_complement(1) == 0
assert answer_next_alternating(5) == 5
assert answer_next_alternating(6) == 10
assert answer_next_alternating(11) == 21
for n in range(1, 3000):
    assert answer_alt_bits_mod(n) == Solution().hasAlternatingBits(n)
    assert (answer_longest_alternating_run(n) == n.bit_length()) == Solution().hasAlternatingBits(n)
print("Đáp án tham khảo: tất cả assert PASS")

Đáp án tham khảo: tất cả assert PASS


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def hasAlternatingBits(self, n: int) -> bool:
        x = n ^ (n >> 1)            # bit xen kẽ -> x toàn bit 1
        return (x & (x + 1)) == 0   # toàn bit 1 <=> x & (x+1) == 0
```

Bản vòng lặp dễ giải thích:
```python
prev = n & 1; n >>= 1
while n:
    if (n & 1) == prev: return False
    prev = n & 1; n >>= 1
return True
```

**Câu thần chú:** *"XOR với chính mình dịch phải; toàn 1 thì xen kẽ."*

**Checklist:**
- [ ] Bỏ `"0b"` khi xử lý chuỗi theo chỉ số.
- [ ] Dùng `>>` hoặc `//`, không dùng `/`.
- [ ] Toàn bit 1 là `x & (x + 1)`, không phải `x & (x - 1)`.
- [ ] Thêm ngoặc quanh biểu thức bit trước khi so sánh `==`.
- [ ] Test `1`, `2`, `3`, `2^31 - 1`.